# Naive Bayes — From Scratch (NumPy only)

**Read `README.md` first.** Every block cites the section (§) it implements.

Third philosophy of the course: no gradients (lessons 01–02), no stored neighbors (lesson 03) — **learning is counting**, and prediction is an evidence meter (README §2.3).

In [1]:
# Block 1 — Imports. NumPy and a dictionary. That's genuinely all this model needs.
import numpy as np
import pandas as pd

np.random.seed(21)

## Block 2 — Look at the data BEFORE modeling *(README §1)*

First text dataset of the course: the features are WORDS. The model's belief: spam and ham have different characteristic vocabularies. Eyeball a few of each.

In [2]:
df = pd.read_csv("data/sms_data.csv")
print(f"{len(df)} messages | balance: {df.label.value_counts().to_dict()}  (imbalanced on purpose)\n")
print("--- three random SPAM ---")
for m in df[df.label=="spam"].sample(3, random_state=1).message: print("  ", m)
print("--- three random HAM ----")
for m in df[df.label=="ham"].sample(3, random_state=1).message: print("  ", m)
print("\nBelief check: the vocabularies clearly differ ✓")

250 messages | balance: {'ham': 160, 'spam': 90}  (imbalanced on purpose)

--- three random SPAM ---
   prize offer buy time cash can discount
   prize now buy urgent free free
   link your your we click link
--- three random HAM ----
   your movie call you is your lunch meeting at
   prize thanks love claim we movie now
   home mom time we dinner the dinner please

Belief check: the vocabularies clearly differ ✓


## Block 3 — Bag of words *(README §0)*

This lesson's preprocessing (where lessons 01–03 had scaling): chop messages into words, build the vocabulary, represent each message as word counts. Order is deliberately thrown away — "claim your prize" = "prize your claim" here.

We split train/test FIRST and build the vocabulary from the training set only — the sealed-exam rule from lesson 03 §9, text edition (README §9, mistake 3).

In [3]:
idx = np.random.permutation(len(df))
cut = int(0.8 * len(df))
tr_df, te_df = df.iloc[idx[:cut]], df.iloc[idx[cut:]]

vocab = sorted(set(w for m in tr_df.message for w in m.split()))   # TRAIN words only
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"train: {len(tr_df)} messages | test: {len(te_df)} | vocabulary: {V} words")

def to_counts(messages):
    """README §0 (bag of words): one row per message, one column per vocab word."""
    Xc = np.zeros((len(messages), V), dtype=int)
    for r, m in enumerate(messages):
        for w in m.split():
            if w in w2i:                      # unseen test words: no column, skipped
                Xc[r, w2i[w]] += 1
    return Xc

Xtr, Xte = to_counts(tr_df.message.values), to_counts(te_df.message.values)
ytr = (tr_df.label.values == "spam").astype(int)
yte = (te_df.label.values == "spam").astype(int)

train: 200 messages | test: 50 | vocabulary: 42 words


## Block 4 — The prior: one line of counting *(README §3.3)*

Before reading a single word, what are the odds a message is spam? Just the base rate. This is where the evidence meter starts (README §2.3).

In [4]:
prior_spam = ytr.mean()
prior_ham  = 1 - prior_spam
print(f"P(spam) = {prior_spam:.2f}   P(ham) = {prior_ham:.2f}")
print("The meter starts tilted toward ham — spam must EARN its verdict with evidence.")

P(spam) = 0.38   P(ham) = 0.62
The meter starts tilted toward ham — spam must EARN its verdict with evidence.


## Block 5 — The likelihood tables: training is complete after this cell *(README §3.3)*

For each word: how often does it appear in each class? Add 1 to every count (Laplace smoothing — README §2.4) so no word can ever veto a class. These two tables + the prior ARE the model.

In [5]:
ALPHA = 1  # smoothing strength — README §4

word_counts_spam = Xtr[ytr==1].sum(axis=0)          # how many times each word appears in spam
word_counts_ham  = Xtr[ytr==0].sum(axis=0)

P_w_spam = (word_counts_spam + ALPHA) / (word_counts_spam.sum() + ALPHA * V)   # README §3.3
P_w_ham  = (word_counts_ham  + ALPHA) / (word_counts_ham.sum()  + ALPHA * V)

# Training is DONE. No loop, no epochs, no learning rate. Compare lesson 01 Block 7 and enjoy the silence.
evidence = np.log(P_w_spam / P_w_ham)               # each word's push on the meter
top = np.argsort(evidence)
print("strongest HAM words :", [vocab[i] for i in top[:6]])
print("strongest SPAM words:", [vocab[i] for i in top[-6:][::-1]])

strongest HAM words : ['see', 'ok', 'movie', 'love', 'lunch', 'call']
strongest SPAM words: ['deal', 'limited', 'urgent', 'click', 'claim', 'offer']


## Block 6 — Prediction: the evidence meter in log-space *(README §3.4)*

Start at the log-prior, ADD each word's log-likelihood (logs turn the fragile product into a safe sum — README §3.4), do it for both classes, pick the higher score.

In [6]:
log_prior = np.array([np.log(prior_ham), np.log(prior_spam)])
log_like  = np.log(np.vstack([P_w_ham, P_w_spam]))          # shape (2, V)

def predict(Xc):
    """README §3.4 — score(c) = log prior + sum of word log-likelihoods; argmax wins."""
    scores = Xc @ log_like.T + log_prior                     # (n_messages, 2)
    return scores.argmax(axis=1), scores

pred, scores = predict(Xte)
print("Example — the first test message:")
print(f"  '{te_df.message.iloc[0]}'")
print(f"  score(ham) = {scores[0,0]:.1f}   score(spam) = {scores[0,1]:.1f}"
      f"   →  predicted: {'spam' if pred[0]==1 else 'ham'}   (truth: {te_df.label.iloc[0]})")

Example — the first test message:
  'ok game thanks now night is please buy love game'
  score(ham) = -36.5   score(spam) = -45.6   →  predicted: ham   (truth: ham)


## Block 7 — Grade it: precision first, this is spam *(README §5)*

A false positive here means a real message thrown into the spam folder — the costly mistake. Same metrics as lessons 02–03; the priorities flip with the use case.

In [7]:
TP = int(np.sum((pred==1)&(yte==1))); FP = int(np.sum((pred==1)&(yte==0)))
TN = int(np.sum((pred==0)&(yte==0))); FN = int(np.sum((pred==0)&(yte==1)))

print("Confusion matrix                predicted HAM   predicted SPAM")
print(f"actually HAM  (real mail)   |      {TN:3d}        |     {FP:3d}  ← real mail lost! the costly one")
print(f"actually SPAM               |      {FN:3d}        |     {TP:3d}")
acc  = (TP+TN)/len(yte)
prec = TP/(TP+FP) if TP+FP else float('nan')
rec  = TP/(TP+FN) if TP+FN else float('nan')
print(f"\nAccuracy  = {acc:.0%}   (baseline 'always ham' would score {1-yte.mean():.0%} — README §9)")
print(f"Precision = {prec:.0%}   ← when it cries SPAM, how often it's right. THE number for spam filters")
print(f"Recall    = {rec:.0%}   ← how much real spam it catches")

Confusion matrix                predicted HAM   predicted SPAM
actually HAM  (real mail)   |       36        |       0  ← real mail lost! the costly one
actually SPAM               |        1        |      13

Accuracy  = 98%   (baseline 'always ham' would score 72% — README §9)
Precision = 100%   ← when it cries SPAM, how often it's right. THE number for spam filters
Recall    = 93%   ← how much real spam it catches


## Block 8 — Break it on purpose: remove the smoothing *(README §2.4)*

Take an obvious spam message, append ONE word never seen in training spam, set ALPHA = 0. Watch a single innocent word veto three words of screaming evidence.

In [8]:
test_msg = "free cash prize zzznewword"   # 3 screaming spam words + 1 word unseen in training

def score_message(msg, alpha):
    Pws = (word_counts_spam + alpha) / (word_counts_spam.sum() + alpha * (V + 1))
    Pwh = (word_counts_ham  + alpha) / (word_counts_ham.sum()  + alpha * (V + 1))
    p_spam, p_ham = prior_spam, prior_ham
    for w in msg.split():
        i = w2i.get(w)
        pw_s = Pws[i] if i is not None else (alpha / (word_counts_spam.sum() + alpha * (V + 1)))
        pw_h = Pwh[i] if i is not None else (alpha / (word_counts_ham.sum()  + alpha * (V + 1)))
        p_spam *= pw_s
        p_ham  *= pw_h
    return p_spam, p_ham

for alpha in [1, 0]:
    ps, ph = score_message(test_msg, alpha)
    verdict = "spam" if ps > ph else ("ham" if ph > ps else "TIE at exactly 0 — model paralyzed")
    print(f"alpha={alpha}:  P(msg|spam)·prior = {ps:.2e}   P(msg|ham)·prior = {ph:.2e}   → {verdict}")

print("\nWith alpha=0 the unseen word contributes probability 0, the product's veto fires,")
print("and BOTH classes score zero — three words of screaming evidence, erased (README §2.4).")

alpha=1:  P(msg|spam)·prior = 1.76e-08   P(msg|ham)·prior = 3.21e-10   → spam
alpha=0:  P(msg|spam)·prior = 0.00e+00   P(msg|ham)·prior = 0.00e+00   → TIE at exactly 0 — model paralyzed

With alpha=0 the unseen word contributes probability 0, the product's veto fires,
and BOTH classes score zero — three words of screaming evidence, erased (README §2.4).


## Block 9 — Explain one prediction: the per-word receipts *(README §2.3)*

Interpretability for free: every word's exact push on the meter. Few models can itemize their verdicts like this.

In [9]:
msg = "free prize claim your lunch now"
running = np.log(prior_spam / prior_ham)
print(f"{'':14s} running log-odds (spam vs ham)")
print(f"{'start(prior)':14s} {running:+.2f}")
for w in msg.split():
    i = w2i[w]
    push = np.log(P_w_spam[i] / P_w_ham[i])
    running += push
    tag = "→ spam" if push > 0 else "→ ham "
    print(f"{w:14s} {running:+.2f}   (this word pushed {push:+.2f} {tag})")
print(f"\nVerdict: {'SPAM' if running > 0 else 'HAM'} — note 'lunch' pushed toward ham and was outvoted (README §2.3)")

               running log-odds (spam vs ham)
start(prior)   -0.49
free           +1.14   (this word pushed +1.63 → spam)
prize          +2.07   (this word pushed +0.94 → spam)
claim          +4.29   (this word pushed +2.22 → spam)
your           +4.52   (this word pushed +0.23 → spam)
lunch          +2.73   (this word pushed -1.79 → ham )
now            +2.34   (this word pushed -0.39 → ham )

Verdict: SPAM — note 'lunch' pushed toward ham and was outvoted (README §2.3)


## Block 10 — Where does it fail? *(README §6)*

Inspect the misclassified test messages. Expect mixed-vocabulary messages — spammy words in innocent contexts and vice versa. That's the naive assumption (and the bag of words) meeting reality.

In [10]:
wrong = np.where(pred != yte)[0]
print(f"{len(wrong)} of {len(yte)} test messages misread:\n")
for i in wrong:
    print(f"  truth={te_df.label.iloc[i]:4s} predicted={'spam' if pred[i]==1 else 'ham':4s} | {te_df.message.iloc[i]}")
print("\nMixed-vocabulary messages — exactly where counting word frequencies (with no sense of")
print("order or context) runs out of signal (README §6).")

1 of 50 test messages misread:

  truth=spam predicted=ham  | meeting ok reply ok buy at

Mixed-vocabulary messages — exactly where counting word frequencies (with no sense of
order or context) runs out of signal (README §6).


---
## What you now own

- Bayes' theorem as a working tool: count the easy direction, flip to the direction you want
- Prior, likelihood, posterior — vocabulary you'll reuse far beyond this model
- Training-as-counting: a model with no loop, no learning rate, and millisecond fits
- The zero-veto and Laplace smoothing; log-space as the fix for underflow
- Per-word explanations: the evidence meter's receipts

**Next:** `naive_bayes_with_library.ipynb` to verify against sklearn — then folder `05-decision-trees`, learning as asking the best questions.